In [ ]:

import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from sklearn.model_selection import train_test_split
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, Input
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
# --- Import new callbacks and layers ---
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau

# --- Import the built-in EfficientNetB0 model and its preprocessor ---
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input

# --- 1. Setup and Constants ---
IMAGE_DIR = "images" 
ANNOT_DIR = "annotations"
IMG_SIZE = (224, 224)
CHECKPOINT_PATH = "efficientnet_finetuned.weights.h5"

# --- 2. Data Loading and Preprocessing ---
print("Starting data loading process...")
X, y_expr, y_valence, y_arousal = [], [], [], []

# Ensure the annotation directory exists before trying to read from it
if not os.path.exists(ANNOT_DIR):
    raise FileNotFoundError(f"Annotation directory not found at: {ANNOT_DIR}")

file_prefixes = sorted(list(set([f.split('_')[0] for f in os.listdir(ANNOT_DIR) if f.endswith('.npy')])))

for prefix in file_prefixes:
    try:
        img_path = os.path.join(IMAGE_DIR, f"{prefix}.jpg")
        expr_path = os.path.join(ANNOT_DIR, f"{prefix}_exp.npy")
        val_path = os.path.join(ANNOT_DIR, f"{prefix}_val.npy")
        aro_path = os.path.join(ANNOT_DIR, f"{prefix}_aro.npy")

        if not all(os.path.exists(p) for p in [img_path, expr_path, val_path, aro_path]):
            continue

        valence = np.load(val_path)
        arousal = np.load(aro_path)

        if valence == -2 or arousal == -2:
            continue
            
        img = load_img(img_path, target_size=IMG_SIZE)
        img_array = img_to_array(img)
        
        # NOTE: Preprocessing is now handled INSIDE the model to work with augmentation
        X.append(img_array) # Append the raw image array
        y_expr.append(np.load(expr_path))
        y_valence.append(valence)
        y_arousal.append(arousal)

    except Exception as e:
        print(f"Skipping {prefix} due to an error: {e}")

# Convert lists to NumPy arrays
X = np.array(X, dtype="float32")
y_expr = np.array(y_expr, dtype="int")
y_va = np.stack((np.array(y_valence, dtype="float32"), np.array(y_arousal, dtype="float32")), axis=1)

print(f"\nDataset Loaded and Filtered. Total samples: {len(X)}")

# --- 3. Data Splitting (Train, Validation, Test) ---
print("Splitting data into training, validation, and test sets...")
X_train_val, X_test, y_expr_train_val, y_expr_test, y_va_train_val, y_va_test = train_test_split(
    X, y_expr, y_va, test_size=0.2, random_state=42, stratify=y_expr
)
X_train, X_val, y_expr_train, y_expr_val, y_va_train, y_va_val = train_test_split(
    X_train_val, y_expr_train_val, y_va_train_val, test_size=0.25, random_state=42, stratify=y_expr_train_val
)
print(f"Training set size: {len(X_train)}")
print(f"Validation set size: {len(X_val)}")
print(f"Test set size: {len(X_test)}")

# --- 4. Build Model with Data Augmentation ---
print("\nBuilding EfficientNetB0 model with data augmentation...")

# Define the data augmentation layers
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomContrast(0.1),
], name="data_augmentation")

# Define the model input
inputs = Input(shape=(224, 224, 3))

# Apply augmentation
x = data_augmentation(inputs)

# Apply EfficientNet's preprocessing
x = preprocess_input(x)

# Load EfficientNetB0 pre-trained on ImageNet
base_model = EfficientNetB0(include_top=False, weights='imagenet', input_tensor=x)

# Set the fine-tuning strategy: unfreeze more layers
base_model.trainable = True
for layer in base_model.layers[:-40]: # Unfreeze the top 40 layers
    layer.trainable = False

# Add custom heads
x_head = base_model.output
x_head = GlobalAveragePooling2D()(x_head)
x_head = Dense(512, activation="relu")(x_head)
x_head = Dropout(0.5)(x_head)
expr_output = Dense(8, activation="softmax", name="expression_output")(x_head)
va_output = Dense(2, activation="tanh", name="va_output")(x_head)

model = Model(inputs=inputs, outputs=[expr_output, va_output])

# --- 5. Compile the Model ---
optimizer = Adam(learning_rate=3e-5) # Slightly higher initial learning rate for the new setup
losses = { "expression_output": "sparse_categorical_crossentropy", "va_output": "mean_squared_error" }
metrics = { "expression_output": "accuracy", "va_output": "mae" }
loss_weights = { "expression_output": 1.0, "va_output": 0.5 }

model.compile(optimizer=optimizer, loss=losses, loss_weights=loss_weights, metrics=metrics)
model.summary()

# --- 6. Train the Model ---
# This checkpoint saves the best model based on validation accuracy
checkpoint_callback = ModelCheckpoint(
    filepath=CHECKPOINT_PATH,
    monitor='val_expression_output_accuracy',
    mode='max',
    save_best_only=True,
    save_weights_only=True,
    verbose=1
)

# This scheduler reduces the learning rate when accuracy plateaus
lr_scheduler = ReduceLROnPlateau(
    monitor='val_expression_output_accuracy',
    factor=0.5, # Reduce LR by half
    patience=3, # wait 3 epochs of no improvement
    min_lr=1e-7,
    verbose=1
)

y_train_dict = {"expression_output": y_expr_train, "va_output": y_va_train}
y_val_dict = {"expression_output": y_expr_val, "va_output": y_va_val}

# Load weights if a checkpoint exists to resume training
if os.path.exists(CHECKPOINT_PATH):
    print(f"\n--- Checkpoint found! Loading weights from {CHECKPOINT_PATH} to resume training. ---")
    model.load_weights(CHECKPOINT_PATH)
else:
    print("\n--- No checkpoint found. Starting training from scratch. ---")


print("\n--- Starting Model Training ---")
history = model.fit(
    X_train, y_train_dict,
    validation_data=(X_val, y_val_dict),
    epochs=50, # Increased epochs for more thorough training
    batch_size=32,
    callbacks=[checkpoint_callback, lr_scheduler] # Add the new scheduler
)


print(f"The best performing model has been saved to: {CHECKPOINT_PATH}")

print("\n--- Evaluating best model on VALIDATION data ---")
model.load_weights(CHECKPOINT_PATH)
results = model.evaluate(X_val, y_val_dict, verbose=0)

print(f"Validation Expression Accuracy: {results[3] * 100:.2f}%")
print(f"Validation Valence-Arousal MAE:  {results[4]:.4f}")
print(f"\nValidation Total Loss:            {results[0]:.4f}")
print(f"Validation Expression Loss:       {results[1]:.4f}")
print(f"Validation Valence-Arousal Loss:  {results[2]:.4f}")




Starting data loading process...

Dataset Loaded and Filtered. Total samples: 3999
Splitting data into training, validation, and test sets...
Training set size: 2399
Validation set size: 800
Test set size: 800

Building EfficientNetB0 model with data augmentation...


Model: "functional_6"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_5       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ data_augmentation   │ (None, 224, 224,  │          0 │ input_layer_5[0]… │
│ (Sequential)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_8         │ (None, 224, 224,  │          0 │ data_augmentatio… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization_4     │ (None, 224, 224,  │          7 │ rescaling_8[0][0] │
│ (Normalization)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_9         │ (None, 224, 224,  │          0 │ normalization_4[… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv_pad       │ (None, 225, 225,  │          0 │ rescaling_9[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 112, 112,  │        864 │ stem_conv_pad[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 112, 112,  │        128 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_activation     │ (None, 112, 112,  │          0 │ stem_bn[0][0]     │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_dwconv      │ (None, 112, 112,  │        288 │ stem_activation[… │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_bn          │ (None, 112, 112,  │        128 │ block1a_dwconv[0… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_activation  │ (None, 112, 112,  │          0 │ block1a_bn[0][0]  │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_squeeze  │ (None, 32)        │          0 │ block1a_activati… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reshape  │ (None, 1, 1, 32)  │          0 │ block1a_se_squee… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reduce   │ (None, 1, 1, 8)   │        264 │ block1a_se_resha… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_expand   │ (None, 1, 1, 32)  │        288 │ block1a_se_reduc… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_excite   │ (None, 112, 112,  │          0 │ block1a_activati

 Total params: 4,710,573 (17.97 MB)

 Trainable params: 2,711,706 (10.34 MB)

 Non-trainable params: 1,998,867 (7.63 MB)


--- Checkpoint found! Loading weights from efficientnet_finetuned.weights.h5 to resume training. ---


c:\Users\Ibrahim\anaconda3\Lib\site-packages\keras\src\saving\saving_lib.py:757: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 52 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))



--- Starting Model Training ---
Epoch 1/50
75/75 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - expression_output_accuracy: 0.4009 - expression_output_loss: 1.6295 - loss: 1.7387 - va_output_loss: 0.2183 - va_output_mae: 0.3729
Epoch 1: val_expression_output_accuracy improved from -inf to 0.36375, saving model to efficientnet_finetuned.weights.h5
75/75 ━━━━━━━━━━━━━━━━━━━━ 204s 2s/step - expression_output_accuracy: 0.4010 - expression_output_loss: 1.6296 - loss: 1.7387 - va_output_loss: 0.2182 - va_output_mae: 0.3728 - val_expression_output_accuracy: 0.3638 - val_expression_output_loss: 1.6756 - val_loss: 1.7622 - val_va_output_loss: 0.1734 - val_va_output_mae: 0.3285 - learning_rate: 3.0000e-05
Epoch 2/50
75/75 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - expression_output_accuracy: 0.4097 - expression_output_loss: 1.5880 - loss: 1.6950 - va_output_loss: 0.2141 - va_output_mae: 0.3666
Epoch 2: val_expression_output_accuracy improved from 0.36375 to 0.37375, saving model to efficientnet_finetuned.weights.h5
